# Pengolahan Data Terstruktur dengan SQL (SQLite) {#sec-modul-3}

::: {.callout-important title="Sub-CPMK Pertemuan Ini"}
Mahasiswa mampu mengoperasikan basis data relasional SQLite menggunakan kueri SQL dasar, menerapkan operasi `JOIN` untuk menghubungkan tabel relasional kelurahan–kecamatan, dan mengintegrasikan kueri SQL ke dalam *DataFrame* Pandas.
:::

## Pendahuluan

[\<masukkan narasi yang menceritakan 'big why' mengelola basis data perkotaan yang relasional di sini\>]{.teks-placeholder style="color: #8a8a8a;"}

## Gambaran Umum Kasus

Dalam skala kota, menyimpan seluruh data dalam satu berkas tunggal menimbulkan redundansi: nama kecamatan yang sama ditulis berulang-ulang pada setiap baris kelurahannya, sehingga boros ruang dan rawan tidak konsisten ketika data dimutakhirkan. Apalagi di Indonesia pemekaran wilayah adalah isu yang sering diangkat ([Fokus: Geliat Wacana Pemekaran Daerah — CNN Indonesia](https://www.cnnindonesia.com/nasional/fokus/geliat-wacana-pemekaran-daerah-5678/all)). Mengubah-ubah data tersebut dalam satu berkas tunggal hanya akan memperumit pembaruan.

Perencana yang mengelola data lintas instansi perlu cara penyimpanan yang lebih tertib: **basis data relasional**. Menggunakan data profil kelurahan Kota Bandar Lampung dari PODES 2021 (BPS), kita akan memindahkan data kelurahan ke dalam basis data SQLite yang telah memuat tabel kecamatan, mengajukan pertanyaan kependudukan dan fasilitas pendidikan menggunakan kueri SQL — dari penyaringan sederhana hingga penggabungan kelurahan dengan kecamatan induknya — lalu menarik hasilnya kembali ke Pandas untuk diolah lebih lanjut.

## Instruksi

Buat notebook baru `modul/modul-03.ipynb` di direktori proyek Anda, sambungkan ke *kernel* `analitika-perkotaan`, dan kerjakan seluruh instruksi di dalamnya.

### Lemari Arsip Digital: Pengenalan SQLite dan Koneksi Database

Sebelum memulai praktik, pahami dulu tiga konsep dasarnya: basis data relasional, SQL, dan SQLite.

#### Apa itu Basis Data Relasional?

**Basis data relasional** (*relational database*) adalah sistem penyimpanan data digital yang mengorganisasi data ke dalam kumpulan tabel dua dimensi (baris dan kolom) yang saling berhubungan — saling **berelasi** — satu sama lain. Model ini pertama kali dirancang oleh Edgar F. Codd pada tahun 1970, dengan tiga karakteristik utama:

- **Struktur tabular** — setiap tabel mewakili satu entitas tertentu; misalnya satu tabel khusus data kelurahan dan satu tabel khusus data kecamatan.

- **Kunci relasi** (*key*) — hubungan antar-tabel diikat oleh kolom pengenal unik yang sama; tabel kelurahan terhubung ke tabel kecamatan melalui kolom kunci wilayah `kode_kec`.

- **Efisiensi dan integritas** — dengan memisahkan entitas ke tabel-tabel yang berelasi, kita menghindari redundansi (penulisan data berulang) dan menjaga data tetap konsisten ketika terjadi pemutakhiran wilayah.

Dalam analisis perkotaan, perencana sering bekerja dengan banyak tabel yang saling berhubungan — data kependudukan, fasilitas, hingga ekonomi lintas proyek dan lintas instansi. Dibandingkan menumpuk semuanya di berkas-berkas CSV terpisah, basis data relasional menawarkan penyimpanan yang efisien, kueri lintas tabel, dan penghindaran redundansi.

#### Apa itu SQL?

**SQL** (*Structured Query Language*) adalah bahasa standar untuk berkomunikasi dengan basis data relasional. Jika basis data adalah **lemari arsip digital** studio PWK, maka SQL adalah bahasa perintah tertib yang kita gunakan untuk meminta petugas arsip melakukan tindakan tertentu — mengambil berkas kelurahan tertentu, mencatat berkas baru, atau merekap fasilitas umum per kecamatan.

Lalu di mana posisinya terhadap Pandas yang baru kita pelajari? Tabel berikut membandingkan keduanya *head-to-head* (diacu dari *SQL for Data Analysis*, Tanimura, 2021):

| Dimensi Perbandingan | SQL | Python (Pandas) |
| :--- | :--- | :--- |
| **Lokasi & beban komputer** | Pengolahan dilakukan langsung oleh sistem basis data di pusat penyimpanan; komputer kita tidak terbebani saat datanya sangat besar. | Pengolahan dilakukan sepenuhnya di memori komputer kita sendiri; komputer bekerja lebih keras jika datanya sangat besar. |
| **Kekakuan struktur data** | Data harus ditata rapi sejak awal dalam tabel-tabel yang saling terhubung — seperti sekat permanen di laci arsip. | Bentuk data lebih fleksibel dan bebas diubah di tengah jalan, tetapi perintah kodenya lebih panjang dan detail. |
| **Mekanisme perulangan data** | Otomatis memproses seluruh baris sekaligus; tidak perlu perintah membaca baris demi baris. | Kadang perlu perintah khusus agar komputer membaca data baris demi baris secara berurutan. |
| **Kelengkapan analisis** | Sangat cepat untuk menyaring, menyatukan, dan merekap tabel, tetapi tidak bisa dipakai membangun model simulasi kota yang rumit. | Ekosistem alat bantunya melimpah: grafik, visualisasi peta, hingga model simulasi masa depan kota. |

Keduanya bukan pesaing, melainkan pasangan kerja: SQL menyiapkan dan merekap data di lemari arsip, Pandas mengolah hasil rekapnya lebih lanjut. Persis alur itulah yang kita jalani pada modul ini.

#### Mengenal SQLite

**SQLite** adalah basis data relasional yang unik karena bersifat ***serverless***. Berdasarkan dokumentasi resmi SQLite.org, SQLite tidak memerlukan proses server latar belakang (*daemon*) ataupun koneksi jaringan: mesin SQLite adalah *library* yang berjalan langsung di dalam proses aplikasi kita (*in-process*). Seluruh isi basis data — berapa pun banyak tabelnya — disimpan dalam **satu file biner tunggal** di *disk* lokal, sehingga bersifat *zero-configuration*, portabel, dan mudah disalin antarfolder.

Di sinilah jawaban teka-teki penting modul ini: **satu file `.sqlite` adalah satu basis data utuh, bukan satu tabel**. Berbeda dengan CSV yang satu file-nya hanya memuat satu tabel, satu file `.sqlite` adalah satu lemari arsip lengkap yang lacinya (tabelnya) bisa lebih dari satu.

::: {.callout-note title="Pengetahuan Komputer"}
### Basis Data Klien-Server vs *Serverless* {.unnumbered .unlisted}

Basis data kelas industri seperti **PostgreSQL** bekerja dengan arsitektur **klien-server**: data dikelola oleh satu program server yang berjalan terus-menerus, dan analis mengaksesnya lewat jaringan memakai akun serta izin akses. Arsitektur ini unggul untuk data besar yang diakses banyak orang serentak — tetapi merepotkan untuk belajar karena menuntut instalasi dan konfigurasi server.

SQLite memangkas semua itu: tanpa server, tanpa akun, cukup satu file. Karena kesederhanaannya, SQLite menjadi basis data yang paling banyak terpasang di dunia — tertanam di ponsel, peramban, hingga berbagai aplikasi sehari-hari.

Satu klasifikasi lagi: SQLite dan PostgreSQL sama-sama berorientasi baris (*row-store*), menyimpan data per baris di *disk* — efisien untuk pencatatan dan pemutakhiran. Kontrasnya adalah basis data analitik berorientasi kolom (*column-store*) seperti BigQuery dan Snowflake, yang menyimpan data per kolom sehingga sangat cepat merekap dataset berskala raksasa.
:::

Praktik kita mulai. Muat *library* `sqlite3` — pustaka standar bawaan Python, tanpa instalasi tambahan — bersama Pandas, lalu buka koneksi ke file `bdl.sqlite` dari direktori dataset yang dibagikan dosen:

In [1]:
import sqlite3
import pandas as pd

path_db = "../dataset-ganjil-2026-2027/bdl.sqlite"
koneksi = sqlite3.connect(path_db)

`sqlite3.connect(...)` membuka jalur komunikasi ke lemari arsip dan menampungnya di variabel `koneksi`. Semua perintah SQL kita selanjutnya akan lewat jalur ini.

::: {.callout-warning title="Penting!"}
### *Path* Salah Tidak Menimbulkan Galat {.unnumbered .unlisted}

Jika *path* yang diberikan ke `sqlite3.connect(...)` keliru, Python **tidak akan protes** — ia justru membuat file basis data baru yang kosong di lokasi keliru tersebut. Maka bila inspeksi pada langkah berikutnya tidak menemukan tabel apa pun, jangan lanjut: periksa kembali `path_db` Anda, hapus file kosong yang telanjur terbuat, lalu ulangi.
:::

#### Inspeksi Isi Basis Data

Karena satu file dapat berisi banyak tabel, kebiasaan pertama saat menerima file SQLite adalah **membuka lemarinya dan membaca daftar lacinya**. SQLite menyediakan tabel sistem bernama `sqlite_master` (nama barunya: `sqlite_schema`) yang mencatat metadata seluruh objek di dalam basis data. Tiga kolom pentingnya: `type` (jenis objek: `table`, `index`, `view`, `trigger`), `name` (nama objek), dan `sql` (pernyataan `CREATE TABLE` asli pembuat objek itu).

Jalankan kueri pertama Anda untuk mendaftar semua tabel:

In [2]:
kueri_daftar_tabel = "SELECT name FROM sqlite_master WHERE type='table'"

daftar_tabel = koneksi.execute(kueri_daftar_tabel).fetchall()
print(daftar_tabel)

[('kecamatan',)]


Bacalah polanya: `koneksi.execute(...)` mengirim kueri ke basis data, dan `.fetchall()` mengambil seluruh baris hasilnya sebagai sebuah *list* (Modul 2) — setiap baris hasil tampil sebagai satu kelompok nilai di dalam tanda kurung. Hasilnya satu baris saja: lemari arsip kita baru berisi satu laci, tabel `kecamatan`.

Untuk membaca struktur kolom sebuah tabel, gunakan perintah sistem `PRAGMA table_info(nama_tabel)`. Ia mengembalikan rincian setiap kolom: nomor urut, nama kolom, tipe data, wajib-tidaknya terisi (`notnull`), nilai bawaan, dan status kunci primer (`pk`). Bentuk kueri standarnya juga tersedia: `SELECT * FROM pragma_table_info('nama_tabel')`.

In [3]:
struktur_kecamatan = koneksi.execute("PRAGMA table_info(kecamatan)").fetchall()
print(struktur_kecamatan)

[(0, 'kode_kec', 'INTEGER', 0, None, 0), (1, 'nama_kec', 'TEXT', 0, None, 0)]


Tabel `kecamatan` berisi dua kolom: `kode_kec` bertipe `INTEGER` (bilangan bulat) dan `nama_kec` bertipe `TEXT` (teks). Sekarang intip isinya — perintah `SELECT` akan kita bedah tuntas pada bagian Kueri Dasar; untuk sekarang cukup baca `SELECT * FROM kecamatan LIMIT 5` sebagai "ambilkan semua kolom dari tabel kecamatan, lima baris pertama saja":

In [4]:
print(koneksi.execute("SELECT * FROM kecamatan LIMIT 5").fetchall())

[(10, 'Teluk Betung Barat'), (11, 'Telukbetung Timur'), (20, 'Teluk Betung Selatan'), (21, 'Bumi Waras'), (30, 'Panjang')]


Kecamatan-kecamatan Kota Bandar Lampung tercatat rapi di laci ini, masing-masing dengan kode wilayah resmi BPS-nya.

::: {.callout-tip title="Pengetahuan VS Code"}
### Inspeksi Visual dengan SQLite Explorer {.unnumbered .unlisted}

File `.sqlite` adalah file biner — jika dibuka langsung di editor, isinya tampak seperti karakter acak. Agar dapat menjelajahinya secara visual, pasang ekstensi **SQLite** (penerbit: alexcvzz) dari panel Extensions VS Code.

Setelah terpasang: buka *Command Palette* (`Ctrl+Shift+P`), jalankan **SQLite: Open Database**, lalu pilih file `bdl.sqlite`. Panel **SQLITE EXPLORER** akan muncul di sisi kiri bawah; bentangkan untuk melihat daftar tabel, dan klik ikon panah di samping nama tabel untuk menampilkan isinya (*Show Table*).

Inilah padanan visual dari kueri `sqlite_master` dan `PRAGMA table_info` yang barusan Anda jalankan — dua cara membaca lemari arsip yang sama.
:::

### Menyusun Laci Data: Membuat Tabel dan Mengisi Data dari Pandas

Laci kelurahan belum ada — kita yang akan menatanya sendiri dari berkas `bdl_kelurahan.csv` yang sudah kita kenal di Modul 2. Baca kembali berkas itu memakai pola `path_data` dan `kolom_pilihan`; kali ini kolom yang dibutuhkan adalah kunci wilayah, identitas kelurahan, kepadatan penduduk, dan fasilitas TK:

In [5]:
path_data = "../dataset-ganjil-2026-2027/bdl_kelurahan.csv"
kolom_pilihan = ["kode_kec", "nama_kec", "nama_des",
                 "kepadatan_pddk", "jml_tk_negeri", "jml_tk_swasta"]

df_kelurahan = pd.read_csv(path_data, usecols=kolom_pilihan)
df_kelurahan.head()

   kode_kec            nama_kec            nama_des  jml_tk_negeri  \
0        10  Teluk Betung Barat              Bakung              0   
1        10  Teluk Betung Barat             Kuripan              0   
2        10  Teluk Betung Barat  Negeri Olok Gading              0   
3        10  Teluk Betung Barat         Sukarame Ii              0   
4        10  Teluk Betung Barat          Batu Putuk              0   

   jml_tk_swasta  kepadatan_pddk  
0              2            6479  
1              0           16321  
2              1            7761  
3              3            1514  
4              1            2214  

Perhatikan kolom `nama_kec`: nama kecamatan yang sama tertulis berulang pada setiap baris kelurahannya. Buktikan skala redundansinya dengan dua alat Modul 2:

In [6]:
print(len(df_kelurahan))
print(df_kelurahan["nama_kec"].nunique())

126
20


Dari 126 baris, hanya ada 20 nama kecamatan unik — artinya setiap nama kecamatan ditulis berulang rata-rata enam kali. Padahal daftar resmi kecamatan sudah terstruktur rapi di tabel `kecamatan`. Menyimpan `nama_kec` sekali lagi di tabel kelurahan berarti memboroskan ruang dan membuka risiko ketidakonsistenan bila ada pemutakhiran nama.

Maka lakukan **normalisasi** sederhana: buang kolom `nama_kec` (cukup kunci `kode_kec` yang tinggal, sebagai pengikat relasi), lalu tulis *DataFrame*-nya menjadi tabel baru bernama `kelurahan` menggunakan `to_sql`:

In [7]:
df_kelurahan = df_kelurahan.drop(columns=["nama_kec"])
df_kelurahan.to_sql("kelurahan", koneksi, if_exists="replace", index=False)

126

Tiga hal terjadi pada perintah `to_sql` itu. Pertama, Pandas membuatkan tabel `kelurahan` di dalam basis data dan menyalin seluruh barisnya — angka yang tampil di bawah sel adalah jumlah baris yang tersalin. Kedua, `index=False` mencegah nomor indeks *DataFrame* ikut tersimpan sebagai kolom tersendiri. Ketiga, `if_exists="replace"` menentukan sikap ketika tabel bernama sama sudah ada: hapus dan tulis ulang.

::: {.callout-warning title="Penting!"}
### Galat `table kelurahan already exists` {.unnumbered .unlisted}

Tanpa `if_exists="replace"`, perilaku bawaan `to_sql` adalah `if_exists="fail"`: eksekusi kedua atas sel yang sama akan gagal dengan galat `ValueError: Table 'kelurahan' already exists`, karena lacinya memang sudah terisi. Ingat ritual **Restart *kernel* & Run All** dari Modul 2 — tanpa `replace`, notebook Anda tidak akan lolos ritual itu. Karena tabel ini milik kita dan sumber kebenarannya adalah berkas CSV, menghapus lalu menulis ulang adalah sikap yang tepat di sini.
:::

#### Verifikasi Hasil Pengisian Data

Buktikan bahwa lemari arsip kini berisi dua laci, lalu periksa struktur laci baru itu. Variabel `kueri_daftar_tabel` masih tersimpan di papan tulis *kernel* — pakai ulang saja:

In [8]:
print(koneksi.execute(kueri_daftar_tabel).fetchall())
print(koneksi.execute("PRAGMA table_info(kelurahan)").fetchall())

[('kecamatan',), ('kelurahan',)]
[(0, 'kode_kec', 'INTEGER', 0, None, 0), (1, 'nama_des', 'TEXT', 0, None, 0), (2, 'jml_tk_negeri', 'INTEGER', 0, None, 0), (3, 'jml_tk_swasta', 'INTEGER', 0, None, 0), (4, 'kepadatan_pddk', 'INTEGER', 0, None, 0)]


Dua tabel dalam satu file — persis seperti janji konsep di awal. Perhatikan pula tipe datanya: `to_sql` menerjemahkan tipe Pandas secara otomatis menjadi `INTEGER` untuk kolom bilangan bulat, `REAL` untuk bilangan desimal, dan `TEXT` untuk teks. Silakan segarkan panel SQLite Explorer Anda (ikon *refresh*) — tabel `kelurahan` kini juga tampil di sana.

::: {.callout-warning title="Penting!"}
### Basis Data Ini Dipakai Lagi di Modul-Modul Berikutnya {.unnumbered .unlisted}

Lemari arsip `bdl.sqlite` yang baru Anda tata bukan sekadar latihan sekali pakai: modul-modul berikutnya akan **membuka kembali basis data ini** sebagai sumber data praktik. Jangan menghapus atau memindahkan file tersebut dari direktori dataset, dan pastikan tabel `kelurahan` benar-benar terbentuk (lolos verifikasi di atas) sebelum melanjutkan. Jika di kemudian hari file ini rusak atau terhapus, tabel `kelurahan` selalu dapat ditata ulang dari berkas CSV sumbernya dengan mengulang bagian ini.
:::

### Kueri Dasar: Mengambil Data dengan `SELECT` dan Menyaring dengan `WHERE`

Lemari sudah tertata; saatnya belajar meminta petugas arsip bekerja. Kueri pengambilan data tersusun dari klausa-klausa: `SELECT` (kolom apa yang diambil), `FROM` (dari tabel mana), `WHERE` (baris mana yang lolos saringan), `ORDER BY` (diurutkan menurut apa; tambahkan `DESC` untuk urutan menurun), dan `LIMIT` (berapa baris maksimal yang dikembalikan). Klausa `WHERE` adalah padanan penyaringan baris Pandas pada Modul 2 — operator perbandingannya (`>`, `<=`, dan kawan-kawannya) bekerja serupa.

Dua bekal tambahan dari Tanimura (2021, Bab 2) untuk menghadapi data lapangan:

1. **Penanganan nilai kosong** — data yang hilang direpresentasikan sebagai `NULL`, dan menyaringnya memakai operator khusus `IS NULL` atau `IS NOT NULL` (bukan `= NULL`). Kolom-kolom PODES yang kita pakai kebetulan terisi lengkap, tetapi saringan ini adalah sabuk pengaman standar ketika bekerja dengan data instansi yang kelengkapannya belum tentu terjamin.

2. **Konversi tipe data** (*type casting*) — SQLite bertipe dinamis; untuk perhitungan yang konsisten kita bisa memaksa tipe tertentu secara eksplisit dengan `CAST(kolom AS tipe)`. Basis data lain seperti PostgreSQL menyediakan jalan pintas berupa operator `::`.

Susun kueri untuk menemukan kelurahan terpadat: kepadatan di atas 10.000 jiwa/km², urut dari yang terpadat, sepuluh teratas. Kueri sepanjang ini tidak nyaman ditulis sebagai satu baris — tampung dalam string multi-baris:

In [9]:
kueri_dasar = """
SELECT
    nama_des,
    kode_kec,
    CAST(kepadatan_pddk AS REAL) AS kepadatan
FROM
    kelurahan
WHERE
    kepadatan_pddk IS NOT NULL
    AND kepadatan_pddk > 10000
ORDER BY
    kepadatan DESC
LIMIT 10
"""

hasil_dasar = koneksi.execute(kueri_dasar).fetchall()
print(hasil_dasar)

[('Kota Karang', 11, 43907.0), ('Tanjung Agung', 40, 37717.0), ('Sawah Lama', 40, 35775.0), ('Sukajawa', 70, 31830.0), ('Sukamenanti', 80, 28790.0), ('Kebon Jeruk', 40, 26592.0), ('Sawah Brebes', 40, 26055.0), ('Kaliawi', 60, 25700.0), ('Teluk Betung', 20, 25011.0), ('Kota Karang Raya', 11, 23796.0)]


Sepuluh kelurahan terpadat Bandar Lampung kini di tangan Anda. Klausa `AS kepadatan` pada `SELECT` memberi nama panggilan (*alias*) untuk kolom hasil — nama itulah yang dikenali `ORDER BY`.

::: {.callout-note title="Konsep Python"}
### String Multi-baris (*Triple-Quoted*) {.unnumbered .unlisted}

Di Modul 1, string selalu kita tulis satu baris dengan sepasang tanda kutip. Python menyediakan bentuk panjangnya: **string multi-baris**, diapit tiga tanda kutip (`"""` di awal dan di akhir). Segala yang berada di antaranya — ganti baris, indentasi, spasi — tersimpan apa adanya sebagai **satu nilai string biasa**.

Itulah sebabnya `kueri_dasar` dapat kita tata bertingkat agar enak dibaca tanpa menimbulkan galat sintaksis: bagi Python ia tetap satu string utuh yang tinggal dikirim ke basis data. Pola ini akan sering kembali setiap kali kita menampung teks panjang di dalam kode.
:::

::: {.callout-note title="Pengetahuan Komputer"}
### KAPITAL, Indentasi, dan Urutan Evaluasi {.unnumbered .unlisted}

Bagi basis data, `select` dan `SELECT` sama saja. Penulisan kata kunci dengan huruf KAPITAL dan pemenggalan klausa dengan indentasi adalah **konvensi gaya** antarmanusia (Tanimura, 2021, Bab 8): mata cepat menangkap kerangka kueri, dan kueri panjang tetap terbaca.

Trivia keduanya lebih dalam: urutan penulisan klausa bukanlah urutan kerjanya. Secara internal, basis data mengevaluasi `FROM` → `WHERE` → `GROUP BY` → `HAVING` → `SELECT` → `DISTINCT` → `ORDER BY` → `LIMIT`. Jadi meski `SELECT` ditulis paling atas, ia justru dievaluasi belakangan — petugas arsip memilih laci dulu, menyaring berkasnya, baru memutuskan kolom mana yang disalin ke laporan.
:::

### Penggabungan Wilayah: Menghubungkan Tabel Kelurahan dan Kecamatan dengan `JOIN`

Hasil kueri barusan hanya memuat `kode_kec` — angka yang tidak bercerita. Nama kecamatannya ada di laci sebelah. Operasi `JOIN` menyatukan berkas dari dua laci menggunakan pengikat kunci relasi: klausa `ON kel.kode_kec = kec.kode_kec` memerintahkan petugas arsip menjodohkan setiap baris kelurahan dengan baris kecamatan yang kode wilayahnya sama.

Dua perkakas penyerta `JOIN`:

- ***Alias* tabel** — `kelurahan AS kel` memberi nama panggilan pendek, sehingga penyebutan kolom lintas tabel (`kel.nama_des`, `kec.nama_kec`) tetap ringkas.

- **Empat jenis `JOIN`** — `INNER JOIN` hanya mengembalikan baris yang berjodoh di kedua tabel; `LEFT JOIN` mempertahankan seluruh baris tabel kiri walau tidak berjodoh; `RIGHT JOIN` kebalikannya; `FULL OUTER JOIN` mempertahankan keduanya. Kita memakai `LEFT JOIN` dengan tabel `kelurahan` di kiri: setiap kelurahan wajib tampil, berjodoh maupun tidak.

Satu kewaspadaan dari Tanimura (2021, Bab 2): `JOIN` yang kuncinya bermasalah dapat menggandakan baris hasil secara diam-diam. Kata kunci `DISTINCT` tepat setelah `SELECT` memerintahkan pembuangan baris hasil yang kembar persis — sekaligus menjadi alat pembuktian: bila jumlah baris hasil ber-`DISTINCT` tidak berkurang, tidak ada penggandaan yang terjadi.

In [10]:
kueri_gabung = """
SELECT DISTINCT
    kel.nama_des,
    kec.nama_kec,
    kel.kepadatan_pddk
FROM
    kelurahan AS kel
LEFT JOIN
    kecamatan AS kec ON kel.kode_kec = kec.kode_kec
ORDER BY
    kepadatan_pddk DESC
LIMIT 10
"""

hasil_gabung = koneksi.execute(kueri_gabung).fetchall()
print(hasil_gabung)

[('Kota Karang', 'Telukbetung Timur', 43907), ('Tanjung Agung', 'Tanjung Karang Timur', 37717), ('Sawah Lama', 'Tanjung Karang Timur', 35775), ('Sukajawa', 'Tanjung Karang Barat', 31830), ('Sukamenanti', 'Kedaton', 28790), ('Kebon Jeruk', 'Tanjung Karang Timur', 26592), ('Sawah Brebes', 'Tanjung Karang Timur', 26055), ('Kaliawi', 'Tanjung Karang Pusat', 25700), ('Teluk Betung', 'Teluk Betung Selatan', 25011), ('Kota Karang Raya', 'Telukbetung Timur', 23796)]


Sekarang setiap kelurahan terpadat tampil bersama nama kecamatan induknya — dua laci menyatu dalam satu laporan.

::: {.callout-note title="Pengetahuan Komputer"}
### Kunci Primer dan Kunci Asing {.unnumbered .unlisted}

Pada tabel `kecamatan`, kolom `kode_kec` berperan sebagai **kunci primer** (*primary key*): pengenal unik yang nilainya pantang kembar — satu kode, satu kecamatan. Pada tabel `kelurahan`, kolom `kode_kec` yang sama berperan sebagai **kunci asing** (*foreign key*): penunjuk ke baris tabel lain yang boleh berulang — banyak kelurahan menunjuk satu kecamatan induk yang sama.

Mengapa mengikat relasi dengan kode, bukan dengan nama? Karena nama itu rapuh: ejaannya bisa berbeda antardokumen (misalnya "Teluk Betung" dan "Telukbetung") dan namanya bisa berganti saat pemekaran wilayah, sedangkan kode wilayah resmi BPS dijaga konsisten. Inilah alasan kolom `nama_kec` aman kita buang dari tabel kelurahan: relasinya sudah dijamin oleh kunci.
:::

### Agregasi Kota: Mengelompokkan Data dengan `GROUP BY`

Perencana jarang berhenti di daftar kelurahan; laporan akhirnya biasanya berupa rekap pada unit analisis kecamatan (Modul 2). Di SQL, rekap seperti itu dibuat dengan klausa `GROUP BY`, yang mengelompokkan baris-baris bernilai sama, lalu **fungsi agregasi** yang merangkum tiap kelompok menjadi satu angka: `AVG` (rata-rata), `SUM` (jumlah total), `COUNT` (cacah baris), `MIN`, dan `MAX`. `value_counts()` yang Anda pakai di Modul 2 sesungguhnya adalah `GROUP BY` + `COUNT` versi Pandas — dan inilah "rekapitulasi per kecamatan yang lebih kaya" yang dijanjikan modul lalu.

Aturan emasnya: **setiap kolom pada `SELECT` yang tidak dibungkus fungsi agregasi wajib dicantumkan di `GROUP BY`** — petugas arsip harus tahu persis menurut apa berkas dikelompokkan.

#### Penanganan Nilai `NULL` dengan `COALESCE`

Satu jebakan aritmetika SQL: angka yang dijumlahkan dengan `NULL` hasilnya `NULL`. Bila salah satu dari `jml_tk_negeri + jml_tk_swasta` kosong, seluruh hasil penjumlahannya ikut kosong. Penangkalnya adalah fungsi `COALESCE(kolom, nilai_cadangan)` (Tanimura, 2021, Bab 2): ia memeriksa nilai dari kiri ke kanan dan mengembalikan yang pertama tidak-`NULL` — sehingga `COALESCE(jml_tk_negeri, 0)` berarti "pakai nilainya, atau `0` bila kosong". Kolom TK kita memang terisi lengkap, tetapi membiasakan `COALESCE` pada penjumlahan lintas kolom adalah disiplin yang menyelamatkan ketika data instansi datang bolong-bolong.

Susun rekapnya: rata-rata jumlah TK (negeri + swasta) per kecamatan, beserta cacah kelurahannya, urut dari yang tertinggi:

In [11]:
kueri_rekap = """
SELECT
    kec.nama_kec,
    COUNT(kel.nama_des) AS jumlah_kelurahan,
    AVG(COALESCE(kel.jml_tk_negeri, 0) + COALESCE(kel.jml_tk_swasta, 0)) AS rata_rata_tk
FROM
    kecamatan AS kec
LEFT JOIN
    kelurahan AS kel ON kec.kode_kec = kel.kode_kec
GROUP BY
    kec.nama_kec
ORDER BY
    rata_rata_tk DESC
"""

hasil_rekap = koneksi.execute(kueri_rekap).fetchall()
print(hasil_rekap)

[('Sukarame', 6, 5.833333333333333), ('Labuhan Ratu', 6, 4.166666666666667), ('Kedamaian', 7, 4.142857142857143), ('Rajabasa', 7, 3.857142857142857), ('Way Halim', 6, 3.0), ('Sukabumi', 7, 2.857142857142857), ('Langkapura', 5, 2.6), ('Teluk Betung Utara', 6, 2.5), ('Kemiling', 9, 2.4444444444444446), ('Tanjung Karang Barat', 7, 2.4285714285714284), ('Kedaton', 7, 2.4285714285714284), ('Panjang', 8, 2.375), ('Teluk Betung Selatan', 6, 2.3333333333333335), ('Tanjung Karang Pusat', 7, 2.2857142857142856), ('Tanjung Senang', 5, 2.0), ('Tanjung Karang Timur', 5, 2.0), ('Telukbetung Timur', 6, 1.8333333333333333), ('Enggal', 6, 1.5), ('Teluk Betung Barat', 5, 1.4), ('Bumi Waras', 5, 1.2)]


Perhatikan arah `JOIN`-nya kita balik — `kecamatan` di kiri — agar seluruh kecamatan pasti tampil di rekap, termasuk seandainya ada kecamatan yang tidak punya baris kelurahan.

Rekap 20 kecamatan itu benar isinya, tetapi bentuknya masih deretan baris polos hasil `fetchall()`: sulit dibaca, dan belum bisa diolah lanjut. Kita butuh jembatan kembali ke perkakas yang sudah kita kuasai.

### Jembatan SQL–Pandas: Membaca Hasil Kueri ke DataFrame Pandas

Jembatan itu bernama `pd.read_sql(kueri, koneksi)`: kirimkan kueri lewat koneksi, dan hasilnya langsung tersaji sebagai *DataFrame* — lengkap dengan nama kolom sesuai *alias* pada `SELECT`. Jalankan ulang `kueri_rekap` yang masih tersimpan di papan tulis *kernel*:

In [12]:
df_rekap = pd.read_sql(kueri_rekap, koneksi)
df_rekap.head(10)

               nama_kec  jumlah_kelurahan  rata_rata_tk
0              Sukarame                 6      5.833333
1          Labuhan Ratu                 6      4.166667
2             Kedamaian                 7      4.142857
3              Rajabasa                 7      3.857143
4             Way Halim                 6      3.000000
5              Sukabumi                 7      2.857143
6            Langkapura                 5      2.600000
7    Teluk Betung Utara                 6      2.500000
8              Kemiling                 9      2.444444
9  Tanjung Karang Barat                 7      2.428571

Bandingkan dengan keluaran `fetchall()` di atas: isinya sama, wujudnya berbeda — kini berupa tabel rapi yang siap disaring, direkap ulang, atau digabung dengan data lain memakai seluruh kemampuan Pandas dari Modul 2. Terbacalah alur lengkap modul ini: **CSV → `to_sql` → basis data SQLite → kueri SQL → `read_sql` → *DataFrame***. Lemari arsip menertibkan penyimpanan; Pandas melanjutkan analisis.

#### Menutup Sesi Kerja Basis Data

Tutup koneksi setiap kali selesai bekerja:

In [13]:
koneksi.close()

::: {.callout-warning title="Penting!"}
### Tutup Koneksi Sebelum Pulang {.unnumbered .unlisted}

Koneksi yang dibiarkan terbuka dapat mengunci file basis data — program lain (termasuk SQLite Explorer) bisa tertolak mengaksesnya, dan penulisan yang terputus di tengah berisiko merusak file. Ibaratnya laci lemari arsip: selesai bekerja, dorong lacinya sampai tertutup. Setelah `koneksi.close()`, variabel `koneksi` tidak dapat dipakai lagi; jika masih ingin mengueri, buka koneksi baru dengan `sqlite3.connect(...)`.
:::

## Latihan Mandiri

Kerjakan di notebook baru `modul/latihan-03.ipynb` dengan *kernel* `analitika-perkotaan`. Alurnya sama persis dengan instruksi di atas; yang berganti hanya fasilitasnya — dari TK ke **SMP**.

1. **Buka dan inspeksi.** Buka koneksi ke `bdl.sqlite` memakai variabel *path*, lalu daftar seluruh tabelnya dengan kueri `sqlite_master`.

2. **Tata ulang laci kelurahan.** Baca `bdl_kelurahan.csv` memakai `path_data` dan `kolom_pilihan` berisi: `kode_kec`, `nama_des`, `jml_smp_negeri`, `jml_smp_swasta`. Tulis ke tabel `kelurahan` dengan `to_sql` — pikirkan sendiri argumen `if_exists` yang tepat, mengingat tabel itu sudah ada.

3. **Saring dengan kueri dasar.** Dalam string multi-baris, susun kueri yang menampilkan kelurahan yang sama sekali belum memiliki SMP (jumlah SMP negeri ditambah swasta sama dengan nol) — amankan penjumlahannya dengan `COALESCE`.

4. **Rekap per kecamatan.** Susun kueri `LEFT JOIN` + `GROUP BY` untuk menghitung rata-rata jumlah SMP (negeri + swasta) per kecamatan, urut dari rata-rata **terendah** sehingga kecamatan yang paling minim fasilitas tampil teratas.

5. **Tarik ke Pandas.** Jalankan kueri rekap tersebut lewat `pd.read_sql`, tampung hasilnya di *DataFrame* `df_latihan`, lalu tampilkan.

6. **Interpretasi.** Di sel teks, tulis dua-tiga kalimat: kecamatan mana yang rata-rata SMP-nya paling rendah dan paling tinggi, dan apa maknanya bagi pemerataan fasilitas pendidikan dasar di Bandar Lampung.

7. **Tutup dan uji.** Tutup koneksi, lalu jalankan **Restart *kernel* & Run All** — seluruh sel harus lolos berurutan.